In [25]:
import os 
import rampwf as rw
import ramphy as rh
from pathlib import Path
import shutil
import pickle as pkl
from ramphy.hyperopt import parse_hyperparameters
from tqdm.notebook import tqdm
import pandas as pd

challenge = "kaggle_synthanic_v1_1_nX"
source_path = Path("/home/gpaolo/ramp-kits")
from pprint import pprint

In [26]:
# Find submissions
submissions = [submission for submission in (source_path / challenge / 'submissions').iterdir() if "_hyperopt_" in str(submission.name)]

In [27]:
dp = 'data_preprocessor_5_llm_text2vec'
hypers = parse_hyperparameters(submission_path=submissions[0], workflow_element_name=dp)
hp_table = {"sub_name": list()}
for hp in hypers:
    hp_table[hp.name] = list()

In [28]:
for sub in tqdm(submissions):
    hp_table['sub_name'].append(sub.name)
    hypers = parse_hyperparameters(submission_path=submissions[0], workflow_element_name=dp)
    for hp in hypers:
        hp_table[hp.name].append(hp.default)

  0%|          | 0/145 [00:00<?, ?it/s]

In [29]:
hp_df = pd.DataFrame.from_dict(hp_table)
hp_df.set_index("sub_name", inplace=True)

In [30]:
hp_df.value_counts()

encoding_mode_llm2vec  fill_value_num_llm2vec  impute_strategy_num_llm2vec  peft_model_llm2vec  pooling_mode_llm2vec
positional             -1.0                    mean                         mntp-supervised     mean                    145
Name: count, dtype: int64

In [31]:
hp_df["encoding_mode_llm2vec"].unique()

array(['positional'], dtype=object)

In [32]:
for sub in submissions:
    if not (sub /'training_output').exists():
        print(sub)


/home/gpaolo/ramp-kits/kaggle_synthanic_v1_1_nX/submissions/lgbm_hyperopt_c21fd58279


In [4]:
from vllm import LLM
import os
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
model = LLM("meta-llama/Meta-Llama-3-8B-Instruct", trust_remote_code=True, tokenizer_mode='slow')

INFO 06-27 13:37:34 llm_engine.py:100] Initializing an LLM engine (v0.4.2) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=meta-llama/Meta-Llama-3-8B-Instruct)


/home/gpaolo/miniforge3/envs/pangu/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


OSError: Can't load tokenizer for 'meta-llama/Meta-Llama-3-8B-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'meta-llama/Meta-Llama-3-8B-Instruct' is the correct path to a directory containing all relevant files for a LlamaTokenizer tokenizer.

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
messages = [{"role": "user", "content": "What is the capital of France?"}]
formatted_prompt =  tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
output = model.generate(formatted_prompt)

OSError: Can't load tokenizer for 'meta-llama/Meta-Llama-3-8B-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'meta-llama/Meta-Llama-3-8B-Instruct' is the correct path to a directory containing all relevant files for a LlamaTokenizerFast tokenizer.